# Benchmark: Pandas vs Polars vs efficient-apriori

Compares execution time and peak memory of all three approaches on a synthetic dataset with Zipf-skewed item distribution.

In [1]:
import pandas as pd
import numpy as np
import time
import tracemalloc
from faker import Faker
from fastapriori import find_associations
from efficient_apriori import apriori

## 1. Generate Dataset (~1M rows, Zipf-skewed)

In [ ]:
fake = Faker()
Faker.seed(42)
np.random.seed(42)

NUM_UNIQUE_ITEMS = 500
NUM_TRANSACTIONS = 200_000

# Realistic item numbers
item_pool = [fake.bothify(text="??####", letters="ABCDEFGHIJKLMNOPQRSTUVWXYZ") for _ in range(NUM_UNIQUE_ITEMS)]

# Zipf-like weights — a few items are very popular, most are rare
weights = 1.0 / np.arange(1, NUM_UNIQUE_ITEMS + 1) ** 0.8
weights /= weights.sum()

rows = []
for txn_id in range(NUM_TRANSACTIONS):
    n_items = min(np.random.poisson(lam=5) + 1, NUM_UNIQUE_ITEMS)
    items = np.random.choice(item_pool, size=n_items, replace=False, p=weights)
    for item in items:
        rows.append((txn_id, item))

df = pd.DataFrame(rows, columns=["Transaction ID", "Item Number"])
print(f"Rows: {len(df):,}")
print(f"Transactions: {df['Transaction ID'].nunique():,}")
print(f"Unique items: {df['Item Number'].nunique():,}")
print(f"Mean items/txn: {len(df) / df['Transaction ID'].nunique():.1f}")

item_counts = df["Item Number"].value_counts()
print(f"\nItem popularity — top 5: {item_counts.head().tolist()}, bottom 5: {item_counts.tail().tolist()}")

In [2]:
df= pd.read_csv("faker.csv")

In [3]:
df.shape

(1199393, 2)

In [4]:
df.head()

,Transaction ID,Item Number
0,0,TD0049
1,0,IJ4006
2,0,AM9667
3,0,IY8331
4,0,OU3432


## 2. Benchmark: fastapriori (Pandas backend)

In [7]:
tracemalloc.start()
t0 = time.perf_counter()

result_pandas = find_associations(
    df, "Transaction ID", "Item Number",
    min_support=0.0, min_confidence=0.02,
    show_progress=True, backend="pandas",
)

t1 = time.perf_counter()
_, peak_pandas = tracemalloc.get_traced_memory()
tracemalloc.stop()

time_pandas = t1 - t0
print(f"\nPandas backend: {time_pandas:.2f}s | peak memory: {peak_pandas / 1024**2:.1f} MB | rules: {len(result_pandas):,}")
result_pandas.head(5)

Counting co-occurrences: 100%|███████████████████████████████████████████████████| 4999/4999 [00:02<00:00, 1765.23it/s]



Pandas backend: 64.46s | peak memory: 2754.2 MB | rules: 180


,item_A,item_B,instances,support,confidence,lift,conviction,leverage,cosine,jaccard
0,AB2203,DY2499,5,0.000025,0.021645,18.037500,1.0209,0.000024,0.0212,0.0107
1,AB6676,GY2643,5,0.000025,0.020833,16.025385,1.0199,0.000023,0.0200,0.0101
2,AD6337,KY8848,5,0.000025,0.021008,17.879149,1.0203,0.000024,0.0211,0.0107
3,AD6337,FO4137,5,0.000025,0.021008,16.285271,1.0201,0.000023,0.0202,0.0102
4,AF1365,TF3290,5,0.000025,0.021186,16.234483,1.0203,0.000023,0.0201,0.0102


## 3. Benchmark: fastapriori (Polars backend)

In [8]:
tracemalloc.start()
t0 = time.perf_counter()

result_polars = find_associations(
    df, "Transaction ID", "Item Number",
    min_support=0.0, min_confidence=0.02,
    backend="polars",
)

t1 = time.perf_counter()
_, peak_polars = tracemalloc.get_traced_memory()
tracemalloc.stop()

time_polars = t1 - t0
print(f"Polars backend: {time_polars:.2f}s | peak memory: {peak_polars / 1024**2:.1f} MB | rules: {len(result_polars):,}")
result_polars.head(5)

Polars backend: 15.12s | peak memory: 2315.5 MB | rules: 180


,item_A,item_B,instances,support,confidence,lift,conviction,leverage,cosine,jaccard
0,ID4472,BA0369,5,0.000025,0.022422,19.755066,1.0218,0.000024,0.0222,0.0112
1,YR5064,ND7704,5,0.000025,0.021739,17.461044,1.0209,0.000024,0.0209,0.0105
2,DE8761,GN4276,5,0.000025,0.021459,17.808299,1.0207,0.000024,0.0211,0.0107
3,GV4311,UV4289,4,0.000020,0.020305,17.354701,1.0195,0.000019,0.0186,0.0094
4,AX5431,LY6820,5,0.000025,0.020161,18.000893,1.0194,0.000024,0.0212,0.0107


## 4. Benchmark: efficient-apriori

In [9]:
# Convert to list-of-tuples format required by efficient-apriori
transactions_list = df.groupby("Transaction ID")["Item Number"].apply(tuple).tolist()
print(f"Prepared {len(transactions_list):,} transactions")

Prepared 200,000 transactions


In [10]:
tracemalloc.start()
t0 = time.perf_counter()

try:
    itemsets, rules = apriori(
        transactions_list,
        min_support=0.0,
        min_confidence=0.02,
        max_length=2,
    )
    t1 = time.perf_counter()
    _, peak_ea = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    time_ea = t1 - t0
    num_rules_ea = len(rules)
    print(f"efficient-apriori: {time_ea:.2f}s | peak memory: {peak_ea / 1024**2:.1f} MB | rules: {num_rules_ea}")
except Exception as e:
    t1 = time.perf_counter()
    _, peak_ea = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    time_ea = t1 - t0
    num_rules_ea = 0
    print(f"efficient-apriori failed after {time_ea:.2f}s: {e}")
    print(f"Peak memory before failure: {peak_ea / 1024**2:.1f} MB")

efficient-apriori: 1117.11s | peak memory: 6255.9 MB | rules: 180


## 5. Results Comparison

In [13]:
comparison = pd.DataFrame({
    "Method": ["fastapriori (pandas)", "fastapriori (polars)", "efficient-apriori"],
    "Time (s)": [round(time_pandas, 2), round(time_polars, 2), round(time_ea, 2)],
    "Peak Memory (MB)": [
        round(peak_pandas / 1024**2, 1),
        round(peak_polars / 1024**2, 1),
        round(peak_ea / 1024**2, 1),
    ],
    "Output Rules": [len(result_pandas), len(result_polars), num_rules_ea],
})

# Speedup relative to slowest
max_time = comparison["Time (s)"].max()
comparison["Speedup vs Slowest"] = round(max_time / comparison["Time (s)"], 1)

comparison

,Method,Time (s),Peak Memory (MB),Output Rules,Speedup vs Slowest
0,fastapriori (pandas),64.46,2754.2,180,17.3
1,fastapriori (polars),15.12,2315.5,180,73.9
2,efficient-apriori,1117.11,6255.9,180,1.0


## 6. Verify Pandas and Polars backends produce consistent results

In [14]:
# Sort both results the same way for comparison
rp_sorted = result_pandas.sort_values(["item_A", "item_B"]).reset_index(drop=True)
rl_sorted = result_polars.sort_values(["item_A", "item_B"]).reset_index(drop=True)

print(f"Pandas rules: {len(rp_sorted):,} | Polars rules: {len(rl_sorted):,}")

# Check rule counts match
assert len(rp_sorted) == len(rl_sorted), f"Rule count mismatch: {len(rp_sorted)} vs {len(rl_sorted)}"

# Check item pairs match
merged = rp_sorted.merge(rl_sorted, on=["item_A", "item_B"], suffixes=("_pd", "_pl"))
assert len(merged) == len(rp_sorted), "Pair mismatch between backends"

# Check metric values are close
for col in ["support", "confidence", "lift"]:
    max_diff = (merged[f"{col}_pd"] - merged[f"{col}_pl"]).abs().max()
    print(f"  {col}: max difference = {max_diff:.8f}")
    assert max_diff < 1e-4, f"{col} values differ too much: {max_diff}"

print("\nAll consistency checks passed!")

Pandas rules: 180 | Polars rules: 180
  support: max difference = 0.00000000
  confidence: max difference = 0.00000000
  lift: max difference = 0.00000000

All consistency checks passed!


In [15]:
rp_sorted

,item_A,item_B,instances,support,confidence,lift,conviction,leverage,cosine,jaccard
0,AB2203,DY2499,5,0.000025,0.021645,18.037500,1.0209,0.000024,0.0212,0.0107
1,AB6676,GY2643,5,0.000025,0.020833,16.025385,1.0199,0.000023,0.0200,0.0101
2,AD6337,FO4137,5,0.000025,0.021008,16.285271,1.0201,0.000023,0.0202,0.0102
3,AD6337,KY8848,5,0.000025,0.021008,17.879149,1.0203,0.000024,0.0211,0.0107
4,AF1365,TF3290,5,0.000025,0.021186,16.234483,1.0203,0.000023,0.0201,0.0102
...,...,...,...,...,...,...,...,...,...,...
175,YV2057,UJ5206,5,0.000025,0.021186,17.655000,1.0204,0.000024,0.0210,0.0106
176,ZH5129,PO5271,5,0.000025,0.020833,16.868826,1.0200,0.000024,0.0205,0.0104
177,ZI4712,LA7309,5,0.000025,0.020080,15.687500,1.0192,0.000023,0.0198,0.0100
178,ZR9686,VL4496,5,0.000025,0.022124,19.931532,1.0215,0.000024,0.0223,0.0113


In [16]:
rl_sorted

,item_A,item_B,instances,support,confidence,lift,conviction,leverage,cosine,jaccard
0,AB2203,DY2499,5,0.000025,0.021645,18.037500,1.0209,0.000024,0.0212,0.0107
1,AB6676,GY2643,5,0.000025,0.020833,16.025385,1.0199,0.000023,0.0200,0.0101
2,AD6337,FO4137,5,0.000025,0.021008,16.285271,1.0201,0.000023,0.0202,0.0102
3,AD6337,KY8848,5,0.000025,0.021008,17.879149,1.0203,0.000024,0.0211,0.0107
4,AF1365,TF3290,5,0.000025,0.021186,16.234483,1.0203,0.000023,0.0201,0.0102
...,...,...,...,...,...,...,...,...,...,...
175,YV2057,UJ5206,5,0.000025,0.021186,17.655000,1.0204,0.000024,0.0210,0.0106
176,ZH5129,PO5271,5,0.000025,0.020833,16.868826,1.0200,0.000024,0.0205,0.0104
177,ZI4712,LA7309,5,0.000025,0.020080,15.687500,1.0192,0.000023,0.0198,0.0100
178,ZR9686,VL4496,5,0.000025,0.022124,19.931532,1.0215,0.000024,0.0223,0.0113


## 7. Analysis

**fastapriori (both backends)** skips candidate generation entirely — it counts co-occurrences directly via Counter+chain (pandas) or self-join (polars), then computes metrics in a single vectorized pass.

**Polars backend** benefits from:
- Multi-threaded lazy evaluation
- Vectorized self-join (no Python-level iteration)
- Apache Arrow memory format

**Pandas backend** benefits from:
- Lower memory footprint (no join expansion)
- Better for dense transactions (many items per transaction)
- Progress bar support via tqdm

**efficient-apriori** uses the classic Apriori algorithm with candidate generation and pruning. With low `min_support` thresholds, pruning is ineffective and it must scan all candidates — making it significantly slower on large datasets with many unique items.